In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go


In [2]:
# Data imports
dry_eigvecs = np.load('../text_files/eigvecs_global.npy')
filled_eigvecs = np.load('../text_files/filled_eigvecs_global.npy')
filled_wet_eigvecs = np.load('../text_files/canal_eigvecs_global.npy')

In [3]:
# --- Data imports ---
dry_eigvecs = np.load('../text_files/eigvecs_global.npy')
filled_eigvecs = np.load('../text_files/filled_eigvecs_global.npy')
canal_eigvecs = np.load('../text_files/canal_eigvecs_global.npy')

# --- MAC function ---
def compute_mac(A, B, n_modes=4):
    MAC = np.zeros((n_modes, n_modes))

    for i in range(n_modes):
        for j in range(n_modes):

            phi_i = np.asarray(A[:, i]).ravel()
            phi_j = np.asarray(B[:, j]).ravel()

            num = np.abs(np.vdot(phi_i, phi_j))**2
            den = np.vdot(phi_i, phi_i) * np.vdot(phi_j, phi_j)

            MAC[i, j] = num / den

    return MAC

# --- Compute MAC matrices ---
MAC_dry_dry = compute_mac(dry_eigvecs, dry_eigvecs)
MAC_dry_filled = compute_mac(dry_eigvecs, filled_eigvecs)
MAC_dry_canal = compute_mac(dry_eigvecs, filled_wet_eigvecs)



def plot_mac(MAC, title, type_mode):
    plt.figure(figsize=(5, 4))
    plt.imshow(MAC, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='MAC value')
    plt.xlabel("type_mode")
    plt.ylabel("Dry modes")
    plt.title(title, fontweight='bold')
    plt.xticks(np.arange(MAC.shape[1]), np.arange(1, MAC.shape[1] + 1))
    plt.yticks(np.arange(MAC.shape[0]), np.arange(1, MAC.shape[0] + 1))

    # ---- Add MAC values inside the cells ----
    for i in range(MAC.shape[0]):
        for j in range(MAC.shape[1]):
            value = MAC[i, j]
            plt.text(
                j, i,                      # (x, y) position
                f"{value:.2f}",            # format to 2 decimals
                ha='center', va='center',
                color='white' if value < 0.5 else 'black',  # contrast
                fontsize=10
            )

    plt.tight_layout()
